# ThreatLens AI — Notebook 07: Attack Relationship Graph (NetworkX)

**Stage in the pipeline:** `Intelligence -> Graph`

### What this notebook does
1. Loads the cleaned dataset
2. Builds a directed graph connecting source IPs to destination IPs, using only attack traffic
3. Ranks the most suspicious nodes (highest out-degree -- one source attacking many targets)
4. Visualizes the graph around the single most suspicious node
5. Extracts a local subgraph for one entity -- the same "click a node to investigate" view the dashboard's Attack Graph page is built around

Needs `source_ip` / `destination_ip` columns -- same honesty note as Notebook 06: if your CSV doesn't have them, `require_ip_columns()` below will explain exactly what to do.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from src.models.graph import require_ip_columns, build_attack_graph, rank_suspicious_nodes, get_entity_subgraph

plt.rcParams["figure.figsize"] = (10, 8)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


In [ ]:
df = pd.read_parquet(PROCESSED_DIR / "cicids2017_cleaned.parquet")
src_col, dst_col = require_ip_columns(df)
print(f"Using '{src_col}' -> '{dst_col}' as graph edges")

## 1. Build the attack graph

Only ATTACK rows become edges -- Normal traffic is excluded on purpose, since the point of this graph is to surface suspicious relationships an analyst should look at, not model the entire network (which would be unreadable and mostly irrelevant noise).


In [ ]:
G = build_attack_graph(df, src_col, dst_col, attack_col="attack_category", max_edges=2000)
print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

## 2. Rank the most suspicious nodes

Ranked by **out-degree** (how many distinct destinations this source attacked) and **betweenness centrality** (how often this node sits on the path between other suspicious nodes). This is the "suspect-IP -> multiple-users/devices -> critical-alert" chain the blueprint describes as the key pattern to surface.


In [ ]:
suspicious = rank_suspicious_nodes(G, top_n=10)
suspicious

## 3. Visualize the full graph

A force-directed (spring) layout, colored by out-degree -- brighter/larger nodes attacked more distinct destinations. With only attack edges included, this should already look meaningfully different from a random network: a small number of hub-like source IPs, not a uniform mesh.


In [ ]:
degrees = dict(G.out_degree())
node_sizes = [80 + degrees.get(n, 0) * 40 for n in G.nodes()]
node_colors = [degrees.get(n, 0) for n in G.nodes()]

pos = nx.spring_layout(G, seed=42, k=0.6)
plt.figure(figsize=(11, 9))
nx.draw_networkx_edges(G, pos, alpha=0.25, edge_color="#00c583", arrows=True, arrowsize=8)
nodes = nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, cmap="viridis")
plt.colorbar(nodes, label="Out-degree (distinct targets attacked)")
plt.title("Attack relationship graph (attack traffic only)")
plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Zoom into the top suspicious entity

This is the notebook version of clicking a node in the dashboard's Attack Graph page -- pulling just that entity's local neighborhood (its direct connections, one hop out) instead of the whole graph, so an analyst can investigate one suspicious IP without visual clutter from everything else.


In [ ]:
top_entity = suspicious.iloc[0]["node"]
sub = get_entity_subgraph(G, top_entity, depth=1)
print(f"Investigating: {top_entity}")
print(f"Local subgraph: {sub.number_of_nodes()} nodes, {sub.number_of_edges()} edges")

sub_pos = nx.spring_layout(sub, seed=42)
plt.figure(figsize=(9, 7))
node_colors_sub = ["#ff4d5a" if n == top_entity else "#00ffa3" for n in sub.nodes()]
nx.draw(sub, sub_pos, with_labels=True, node_color=node_colors_sub, node_size=800,
        font_size=7, edge_color="#7f9c93", arrows=True)
plt.title(f"Local neighborhood — {top_entity}")
plt.tight_layout()
plt.show()

### Attack types observed on each edge from this entity

Each edge in the graph carries the set of attack types and the flow count seen between that pair -- this is what would populate a "connected users, devices and sessions" detail panel when an analyst clicks this node in the live dashboard.


In [ ]:
edge_details = []
for u, v, data in sub.edges(data=True):
    if u == top_entity:
        edge_details.append({
            "target": v,
            "attack_types": ", ".join(sorted(data["attack_types"])),
            "flow_count": data["weight"],
        })
pd.DataFrame(edge_details).sort_values("flow_count", ascending=False)

## 5. Summary & next steps

| Item | Result |
|---|---|
| Total graph size | see Section 1 |
| Top suspicious entity | see Section 2 |
| Local subgraph size | see Section 4 |

**Next up (`08_threat_forecasting.ipynb`)** shifts from "what's connected to what" to "what's coming next" -- aggregating events into hourly time windows and forecasting 1h/6h/24h threat probability, completing the blueprint's full Intelligence layer (UBA done, Graph done, Threat Score done, SHAP done — Forecasting is the last piece).
